# 🤖 03. Modelagem, Validação Cruzada Estratificada & Otimização Bayesiana
### **Tech Challenge - Fase 3: Predição de Alfabetização Infantil**

Este notebook aborda o terceiro passo do ciclo de Machine Learning:
1. **Definição dos Modelos Candidatos:** Baseline (Regressão Logística L2) vs Ensembles (Random Forest, XGBoost e LightGBM);
2. **Validação Cruzada Estratificada (Stratified 5-Fold CV)** para estimar a capacidade de generalização e mitigar overfitting;
3. **Otimização Bayesiana de Hiperparâmetros via Optuna** no modelo campeão;
4. **Persistência dos modelos treinados** (`.joblib`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import RANDOM_STATE
from src.data_loader import load_gold_silver_data
from src.preprocessing import split_and_preprocess_data
from src.models import get_candidate_models, evaluate_models_cross_validation, fit_and_save_all_models
from src.tuning import tune_lightgbm_optuna

print('Módulos carregados com sucesso!')

## 1. Preparação dos Dados Pré-processados

In [ ]:
df_raw = load_gold_silver_data(sample_size=30000, seed=RANDOM_STATE)
X_train, X_test, y_train, y_test, feature_names, preprocessor = split_and_preprocess_data(df_raw)
print(f'Treino: {X_train.shape[0]:,} amostras | Teste: {X_test.shape[0]:,} amostras')

## 2. Validação Cruzada Estratificada (5-Fold CV) de Todos os Modelos

In [ ]:
candidate_models = get_candidate_models()
cv_comparison = evaluate_models_cross_validation(candidate_models, X_train, y_train)
display(cv_comparison)

## 3. Ajuste Final e Persistência dos Modelos

In [ ]:
trained_models = fit_and_save_all_models(candidate_models, X_train, y_train)

## 4. Otimização Bayesiana de Hiperparâmetros (Optuna)

In [ ]:
best_lgbm, best_params, best_cv_score = tune_lightgbm_optuna(X_train, y_train, n_trials=25)
trained_models['LightGBM_Optimized'] = best_lgbm
print(f'Melhor ROC-AUC CV Optuna: {best_cv_score:.4f}')